# Workshop 3 - Computational Modeling of Biological Systems with ODEs

In this notebook, we will explore how **ordinary differential equations (ODEs)** can be used to simulate **dynamic biological processes** such as gene regulation and signalling

---

## Why ODEs in Biology?
Many biological processes evolve **continuously over time**. ODEs provide a mathematical framework to describe how system states (e.g., concentrations of proteins) change due to:
- **Production** (e.g., transcription, translation, synthesis)
- **Degradation** (e.g., decay, dilution, turnover)
- **Interactions** (e.g., activation, inhibition, feedback loops)

---

## Learning Goals
By the end of this notebook, you will:
1. Understand the basic principles of modeling biological systems with ODEs.
2. Implement and solve ODE models using Python.
3. Explore how parameters influence system dynamics.
4. Simulate a simple **gene regulation circuit** that acts as a biological oscillator or clock.

---

## Tools
We will use:
- `numpy` for numerical operations
- `matplotlib` and `seaborn` for visualizing time courses

---

Let’s begin by importing required python packages


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Next, let's review a simple biological process and translate it into an ODE model. Assume we have a particular gene in a cell, and our model tracks the number of protein copies of this gene over time.

## A Simple ODE: Constant Protein Production

Let's start with a simple example:  
Imagine a cell starts with 20 copies of a protein P, and produces a protein at a constant rate of **1 copy per minute**, and for now we will **ignore degradation**.

---

### Defining the variable

- Let $P(t)$ be the number of protein molecules at time $t$ (in minutes).
- The **rate of change** of protein concentration is written as:

$$
\frac{dP}{dt} = 1
$$

This means:  
> *At every moment in time, the cell is producing proteins at a rate of 1 copy per minute.*

---

### Solving the ODE

This ODE is simple enough to solve analytically:

$$
\frac{dP}{dt} = 1 \quad \Rightarrow \quad P(t) = t + C
$$

- Here, $C$ is the constant of integration.  
- If we assume we start with **20 protein copies at time $t=0$**, then $P(0)=20 \;\Rightarrow\; C=20$.

So the solution becomes:

$$
P(t) = t + 20
$$

---
Let's write some code to visualize this solution



In [ ]:
# Time values
t = ## FILL IN - use numpy's linspace to create an array from 0 to 100 (hint, there will be 101 entries in this array)

# Implement the function P(t)
P = ## FILL IN - use the above solution for P as a function of t

# Plot
plt.figure(figsize=(8, 5))
sns.lineplot(x=t, y=P, label="P(t)")
plt.xlabel("Time (minutes)")
plt.ylabel("Protein copies")
plt.title("Plot of P(t)")
plt.grid(True)
plt.legend()
plt.show()

## Protein Degradation with a First-Order Rate

Previously, we considered a cell that produced proteins at a constant rate.  
Now, let’s flip the situation: suppose there is **no new protein production**,  
and instead proteins are removed (degraded) at a **constant fractional rate**.
We still start with 100 proteins in the cell.

---

### Biological Assumptions
- At every minute, about **5% of the protein molecules** are degraded.  
- This is an example of **first-order degradation**, where the rate of decay is proportional to the current amount.

---

### Differential Equation
Let $P(t)$ be the number of protein molecules at time $t$ (in minutes).  
The rate of change is given by:

$$
\frac{dP}{dt} = -kP
$$

where:
- $k$ is the degradation rate constant.  
- Here, $k = 0.05 \ \text{per minute}$ (since 5% of proteins degrade each minute).

---

### Analytical Solution
The solution to this ODE is the familiar exponential decay function:

$$
P(t) = P(0) \, e^{-kt}
$$

- $P(0)$ is the initial protein number at time $t=0$.  
- Over time, the protein amount decreases exponentially toward zero.

At $t=0$, $P(0) = 20$, so we have:
$$
P(t) = 20 \, e^{-0.05t}
$$

where $t$ is in minutes.

---
Again, lets write some code to plot this solution

In [ ]:
# Time values
t = ## FILL IN - use numpy's linspace to create an array from 0 to 100 (hint, there will be 101 entries in this array)

# Implement the function P(t)
P = ## FILL IN - use the above solution for P as a function of t

# Plot
plt.figure(figsize=(8, 5))
sns.lineplot(x=t, y=P, label="P(t)")
plt.xlabel("Time (minutes)")
plt.ylabel("Protein copies")
plt.title("Plot of P(t)")
plt.grid(True)
plt.legend()
plt.show()

Note these results are a little funny in that they allow fractional copies of a protein to exist in a cell. We can hand-wave around this by saying our model describes the average number of proteins found in each cell within a large population. Or perhaps instead of absolute protein copy count, P represents protein concentration. But for now let's not get bogged down.

### Solving the ODE numerically
So far, the ODEs we've looked at have been simple enough to solve analytically. However, this is not always possible. Instead of using an analytical solution, we can also solve ODEs numerically. This is known as **numerical integration**, and one of the simplest algorithm's for numerical integration in known as **Euler's method**

### Euler’s Method for Numerical Integration

Given an ODE of the form:

$$
\frac{dP}{dt} = f(t, P), \quad P(0) = P_0
$$

Euler’s method uses a small time step $\Delta t$ to iteratively update the solution (Euler update formula):

$$
P_{n+1} = P_n + \Delta t \cdot f(t_n, P_n)
$$

- $P_n$ is the current estimate of $P(t)$  
- $f(t_n, P_n)$ is the slope (rate of change) given by the ODE  
- $\Delta t$ is the step size (smaller steps → higher accuracy, but more computation)

---

### Intuition
- Start from the initial condition $P_0$ at $t=0$.  
- Use the derivative (slope) at that point to take a small “step” forward.  
- For instance, if we set $\Delta t=1s$, then we will calculate $P(t+1)$=$P(t)+1s*\frac{dP}{dt}(t)$
- Repeat this process for $P(t+1),P(t+2),P(t+3)...$ to trace out the approximate trajectory of $P(t)$.  

Over many steps, the sequence of updates builds an approximation of the true solution.

---

![Eulers method GIF](euler.gif "Euler's Method")

---
Let's now write some code using a FOR loop to implement Euler's Method to solve the P(t) with degradation as described above.

In [ ]:
# Parameters
P0 = 20       # initial protein count
k = 0.05       # degradation rate per minute
dt = 1.0       # time step (minutes)
n_steps = 100    # total number of steps to simulate (in this case 100 steps = 100 minutes)

# Arrays to store results
t = np.zeros(n_steps + 1)
P = np.zeros(n_steps + 1)

# Initial condition
P[0] = P0

# Euler's method loop
for i in range(n_steps):
    dP_dt = ## FILL IN - use the above expression for dP/dt as a function of P
    P[i+1] = ## FILL IN - use the Euler update formula
    t[i+1] = t[i] + dt                 # update time

# Analytical solution for comparison
P_exact = P0 * np.exp(-k * t)

# Seaborn styling
sns.set(style="whitegrid", context="talk")

# Plot results
plt.figure(figsize=(9, 6))
sns.lineplot(x=t, y=P, marker="o", label="Euler approximation")
sns.lineplot(x=t, y=P_exact, linestyle="--", label="Exact analytical solution")
plt.xlabel("Time (minutes)")
plt.ylabel("Protein copies")
plt.title("Protein Degradation (Euler Method vs. Exact Solution)")
plt.legend()
plt.show()

Note there is a slight divergence between the exact solution (using calculus to solve the differential equation) and our numerical simulation, but it isn't too far off! The divergence is due to the fact that we use a $\Delta t>0$. If we decrease $\Delta t$ (e.g., to less than 1 minute), accuracy will improve at the expense of increased calculation time. This is pretty cool in that we can use numerical integration to solve ODEs with pretty good accuracy, without doing any any real calculus. Moreover, this methods works even if the ODEs can't be solved analytically. Let's take a look at such an example by exploring a system with **negative feedback**

## Protein Production with Negative Feedback

In real biological systems, protein production is often **regulated by feedback mechanisms**.  
One common form of regulation is **negative feedback**, where the protein itself **inhibits its own production**.

![Negative feedback motif](negFeedback.png "Negative feedback motif")

---

### Model Description

Let $P(t)$ represent the number of protein molecules at time $t$ (in minutes).  

We include two processes:
1. **Production with negative feedback**: modeled using a **Hill function**  
2. **First-order degradation**: as before

The ODE becomes:

$$
\frac{dP}{dt} = \underbrace{\frac{\alpha}{1 + \left(\frac{P}{P_\text{half}}\right)^n}}_{\text{production with negative feedback}} - \underbrace{k P}_{\text{degradation}}
$$

Where:
- $\alpha$ = 1 = maximum production rate (copies/min). 
- $P_\text{half} = 10$ = protein level at which production is half-maximal  
- $n = 2$ = Hill coefficient controlling feedback response  
- $k = 0.05$ = first-order degradation rate per minute

Note as in our very first example, protein can be produced at 1 copy/min, but now this only occurs if there is no protein to begin with (P=0). Otherwise, as P increases, the actual production rate drops according to the Hill function term $\frac{1}{1 + \left(\frac{P}{10}\right)^2}$

---

To get a better sense, let's plot the shape of this Hill Function


In [ ]:
# Parameters for Hill function
alpha = 1.0       # max production rate (arbitrary units)
P_half = 10       # half-max protein concentration
n = 2             # Hill coefficient

# Protein levels to evaluate
P_values = np.linspace(0, 30, 300)

# Hill function: production vs protein
production = alpha / (1 + (P_values / P_half)**n)

# Plot
plt.figure(figsize=(8,5))
sns.lineplot(x=P_values, y=production)
plt.xlabel("Protein count P")
plt.ylabel("Production rate (copies/min) ")
plt.title("Negative Feedback Production: Hill Function")
plt.axhline(alpha, color='gray', linestyle='--', alpha=0.5, label="Max production")
plt.axvline(P_half, color='red', linestyle='--', alpha=0.5, label="P_half")
plt.legend()
plt.show()


### Hill Function Intuition

- When $P \ll P_\text{half}$, production is near maximal ($\approx \alpha$).  
- When $P \gg P_\text{half}$, production is strongly inhibited ($\approx 0$).  
- $n$ controls how sharply production drops near $P_\text{half}$.  




### Full ODE

Substituting the values for all our parameters, we end up with the following

$$
\frac{dP}{dt} = \frac{\alpha}{1 + (P/10)^2} - 0.05 P
$$

This is a **nonlinear ODE**. Unlike the linear degradation-only example, it generally **cannot be solved analytically**, so we will use Euler’s method.



In [ ]:
# Parameters
alpha = 1.0        # max production rate (copies/min)
P_half = 10        # half-max protein concentration
n = 2              # Hill coefficient
k = 0.05           # degradation rate per minute
dt = 1.0           # time step (minutes)
P0 = 20       # initial protein count
n_steps = 100    # total number of steps to simulate (in this case 100 steps = 100 minutes)

# Arrays to store results
t = np.zeros(n_steps + 1)
P = np.zeros(n_steps + 1)
P[0] = P0

# Euler's method loop
for i in range(n_steps):
    # Hill function production term
    production = ## FILL IN - use the above expression for Hill function regulating production
    
    # Degradation term
    degradation = ## FILL IN - use the above expression first order degradation with rate k
    
    # Euler update
    dP_dt = ## FILL IN - what is the change in P as a function of production and degradation?
    P[i+1] = ## FILL IN - use the Euler update formula
    
    # Update time
    t[i+1] = t[i] + dt

# Plot results
plt.figure(figsize=(9,6))
sns.lineplot(x=t, y=P)
plt.xlabel("Time (minutes)")
plt.ylabel("Protein copies")
plt.title("Protein Dynamics with Negative Feedback and Degradation (Euler Method)")
plt.grid(True)
plt.show()


As before, we see an exponential decay, but now it asymptotes above zero toward a steady state P of around 10. 

## Explore the System by Changing Parameters

Now that we have implemented a model for **protein production with negative feedback and degradation**, you can experiment with different parameters to see how the system behaves. I would recommend duplicating the above cell prior to making these changes so you always have a copy of the original.

### Suggested parameters to vary:
- $\alpha$ — maximum production rate  
- $P_\text{half}$ — protein concentration at half-max production  
- $n$ — Hill coefficient (controls steepness of feedback)  
- $k$ — degradation rate  

### Things to observe:
- How does changing $\alpha$ affect the **steady-state protein level**?  
- How does increasing $n$ change the **sharpness of the feedback**?  
- How does varying $P_\text{half}$ shift the **feedback threshold**?  
- How does changing $k$ influence the **balance between production and degradation**?  


In [ ]:
## FILL IN - duplicate above code cell here and explore effects of different parameters

## Extending to a Three-Protein System: P, Q, and R

We can expand our model to simulate multiple proteins simultaneously.  
Each protein is regulated by **negative feedback** with the same Hill parameters, and undergoes first-order degradation.

### Model Setup

For proteins $P$, $Q$, and $R$:

$$
\frac{dP}{dt} = \frac{\alpha}{1 + (P/P_\text{half})^n} - k P
$$

$$
\frac{dQ}{dt} = \frac{\alpha}{1 + (Q/P_\text{half})^n} - k Q
$$

$$
\frac{dR}{dt} = \frac{\alpha}{1 + (R/P_\text{half})^n} - k R
$$

Where:
- $\alpha = 1.0$ — maximum production rate  
- $P_\text{half} = 10$ — half-max protein level  
- $n = 2$ — Hill coefficient controlling feedback steepness  
- $k = 0.05$ — degradation rate per minute

---

### Initial Conditions
- $P(0) = 20$  
- $Q(0) = 0$  
- $R(0) = 5$

---

Let's now implement a **numerical simulation** (Euler's method) for all three proteins simultaneously


In [ ]:
# Parameters (same for all proteins)
alpha = 1.0       # max production rate (copies/min)
P_half = 10       # half-max protein concentration
n = 2             # Hill coefficient
k = 0.05          # degradation rate per minute
dt = 1.0          # time step (minutes)
n_steps = 100    # total number of steps to simulate (in this case 100 steps = 100 minutes)

# Initial conditions
P0, Q0, R0 = 20.0, 0.0, 5.0

# Arrays to store results
t = np.zeros(n_steps + 1)
P = np.zeros(n_steps + 1)
Q = np.zeros(n_steps + 1)
R = np.zeros(n_steps + 1)

# Set initial values
P[0], Q[0], R[0] = P0, Q0, R0

# Euler's method loop
for i in range(n_steps):
    ## FILL IN production terms for P, Q, R
    prod_P = 
    prod_Q = 
    prod_R = 
    
    ## FILL IN degradation terms for P, Q, R
    deg_P = 
    deg_Q = 
    deg_R = 
    
    ## FILL IN production terms for P, Q, R
    P[i+1] = 
    Q[i+1] = 
    R[i+1] = 
    
    # Update time
    t[i+1] = t[i] + dt

# Plot results
plt.figure(figsize=(10,6))
sns.lineplot(x=t, y=P, label="P (start=100)")
sns.lineplot(x=t, y=Q, label="Q (start=0)")
sns.lineplot(x=t, y=R, label="R (start=5)")
plt.xlabel("Time (minutes)")
plt.ylabel("Protein copies")
plt.title("Dynamics of Proteins P, Q, and R with Negative Feedback and Degradation")
plt.legend()
plt.show()


## Modify the System to Create a Repressor Loop

Currently, each protein represses **its own production** via negative feedback.  
A more complex and biologically interesting scenario is a **repressor loop**, where proteins inhibit each other in sequence.

### Goal

- **P represses Q**  
- **Q represses R**  
- **R represses P** (or you can choose R represses Q to form a closed loop)  

This creates a **regulatory loop** rather than independent self-repression.

---

### Steps for Modification

1. Duplicate the code cell above into a new cell below. In the simulation code:
   - Replace the self-repression Hill functions with **cross-repression**
   - Keep the **degradation term** as before: e.g., degradation rate of P still depends on P.
3. Experiment with initial protein concentrations and parameters.

---

Challenge: Can you find parameters where the system exhibits **oscillatory behavior** and parameters where it doesn't? You may need to extend the number of steps to be able to see stable oscillatory behaviour


In [ ]:
## FILL IN duplicate above code cell here and make required modifications

## Congratulations! You’ve Modeled a Simple Biological Oscillator

By creating a repressor loop where proteins inhibit each other, you have implemented a **simple biological oscillator** in silico.  

This type of system is not just theoretical—**bioengineers have built one in reality**, known as the **Repressilator**.  
- The Repressilator consists of three genes arranged in a negative-feedback loop.  
- It produces **oscillatory protein expression** over time, demonstrating how synthetic biology can design dynamic circuits in living cells.  
- Original publication: [Elowitz & Leibler, 2000, Nature](https://www.nature.com/articles/35002125)

### Takeaways
- Simple rules of protein production, degradation, and feedback can generate **complex temporal dynamics**.  
- Modeling these systems with ODEs and numerical methods like Euler’s method allows us to **predict and explore biological behavior** before experimental implementation.  
- Such models form the foundation for designing **synthetic gene circuits** and understanding natural regulatory networks.

### Note on numerical integration
- While we implemented Euler’s method **from scratch**, there are Python packages, such as **SciPy’s `solve_ivp`**, that solve ODEs **more efficiently and accurately**.  
    - They use **adaptive step sizes**, adjusting $\Delta t$ to maintain accuracy without unnecessary computation.  
    - They implement **higher-order methods** (like second, third and even higher derivatives), to reduce numerical error compared to Euler’s simple first-order derivative method.  
